# phase 2 : préparer et construire le modèle.<br>
## Tâche 1 : découper en entraînement et test, dans l’ordre du temps


### importation de data et supprime de redoublant

In [2]:
from sklearn.preprocessing import StandardScaler
import numpy as np 
import pandas as pd

In [3]:
df=pd.read_csv("../data/raw/creditcard.csv")
df.duplicated().sum()
df_clean=df.drop_duplicates()
df_clean.duplicated().sum()

np.int64(0)

### decoupage chronologique


In [4]:
data=df_clean.sort_values("Time")
copure=int(len(data)*0.8) 
train=data.iloc[:copure] # pour que 80% des donnes de entrainement 
test=data.iloc[copure:] #pour que 20% de donner de test
train["Class"].sum()+ test["Class"].sum()

np.int64(473)

In [5]:
len(test)+len(train)

283726

In [6]:
print(train["Time"].max() <= test["Time"].min())

True


## tâche 2 : mettre Amount à l’échelle, sans fuite de données
`1. Le contexte`

Les colonnes V1 à V28 sont déjà sur des échelles comparables (elles viennent d’une ACP). Amount, elle, va de 0 à 25 691 avec une médiane de 22 : quelques gros montants écrasent tout le reste. Dans Time, on a décidé de ne pas utiliser le brut.

In [7]:
df_clean["Amount"].describe()

count    283726.000000
mean         88.472687
std         250.399437
min           0.000000
25%           5.600000
50%          22.000000
75%          77.510000
max       25691.160000
Name: Amount, dtype: float64

moyenne tres superieur a la mediane (50%) :sine d'étirement  88>>>22

In [8]:
df_clean["Amount"].skew()   

np.float64(16.978803370060476)

coffecient d'asymetrie : 16>>1 tres étiré 

In [9]:
# 1. l'algorihtme: sans apprentissage donc sur train et test 
train["Amount_log"] = np.log1p(train["Amount"])
test["Amount_log"] = np.log1p(test["Amount"])


In [10]:
# 2.standarisation : fit sur train seulement 
scaler=StandardScaler()
train["Amount_s"] = scaler.fit_transform(train[["Amount_log"]])
test["Amount_s"]=scaler.transform(test[["Amount_log"]])
train[["Amount","Amount_log","Amount_s"]].head(15)
#La valeur standardisée se lit comme un nombre d’écarts-types par rapport à la moyenne (du train, en échelle logarithmique) :

#0 : montant dans la moyenne ;
#positif : au-dessus de la moyenne (ligne 0, 149,62 → +1,10) ;
#négatif : en dessous (ligne 1, 2,69 → −1,13) ;
#au-delà de ±3 : valeur rare ou extrême.

,Amount,Amount_log,Amount_s
0,149.62,5.014760,1.099723
1,2.69,1.305626,-1.134338
2,378.66,5.939276,1.656571
3,123.50,4.824306,0.985010
4,69.99,4.262539,0.646650
5,3.67,1.541159,-0.992474
6,4.99,1.790091,-0.842538
7,40.80,3.732896,0.327639
8,93.20,4.545420,0.817033
9,3.68,1.543298,-0.991185


In [11]:
train["Amount_s"].describe()

count    2.269800e+05
mean     2.444232e-16
std      1.000002e+00
min     -1.920734e+00
25%     -7.486866e-01
50%      5.623500e-03
75%      7.261014e-01
max      4.033866e+00
Name: Amount_s, dtype: float64

In [12]:
print(train["Amount"].skew(), train["Amount_log"].skew())

14.069984483679008 0.13559041406977693


##### Tâche 3 : séparer les variables (X) et la cible (y)

X : les informations qu’il utilise pour prédire (les colonnes).<br>
y : la réponse à retrouver (ici Class, 0 normale ou 1 fraude).<br>
* La cible n’est jamais dans X. Sinon le modèle « lit la réponse » et obtient un score parfait, qui ne vaut rien en vrai. C’est une fuite de données.
* On retire les versions brutes remplacées. Amount et Amount_log ont été transformées en Amount_s : on garde seulement Amount_s, pour ne pas dupliquer l’information.
* On retire Time. Il ne se répète pas d’un jour à l’autre (décision de l’étape 9). On l’a gardé uniquement pour découper.
* Hour est une variable candidate. On va la préparer à part, pour comparer le modèle avec et sans elle.

In [13]:
colonnes=[f"V{i}" for i in range (1,29)]+["Amount_s"]
X_train=train[colonnes]
Y_train=train["Class"]
X_test=test[colonnes]
Y_test=test["Class"]


In [14]:
# controles
list(X_train.columns) == list(X_test.columns)
Y_train.sum() + Y_test.sum()
X_train.isna().sum().sum()
X_train.shape[1]

29

#### Tâche 4 : entraîner un modèle de base et voir pourquoi l’accuracy trompe
* 1. Le contexte

Un modèle de base (baseline) est le modèle le plus simple possible. Il sert de point de comparaison : tout modèle plus compliqué doit faire mieux que lui, sinon il ne sert à rien. On commence par la régression logistique, qui calcule une probabilité de fraude à partir des colonnes de X.

* 2. Le pourquoi : pourquoi le score le plus connu ne suffit pas

L’accuracy est la proportion de prédictions correctes. Avec 0,17 % de fraudes, un « modèle » qui répond toujours « normale » se trompe seulement sur les fraudes : son accuracy est d’environ 99,8 %, alors qu’il ne détecte aucune fraude. Un score élevé peut donc cacher un modèle inutile. Voilà pourquoi on calcule l’accuracy ici, pour la comparer à ce « modèle qui répond toujours 0 », puis on utilisera de meilleures mesures à la tâche 5.

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

modele = LogisticRegression(max_iter=1000)   # 1. créer
modele.fit(X_train, Y_train)                 # 2. apprendre sur le train
Y_pred = modele.predict(X_test)              # 3. prédire sur le test

accuracy_score(Y_test, Y_pred)*100            # proportion de bonnes prédictions



99.93303492757198

In [16]:
Y_zero = np.zeros(len(Y_test))               # modèle idiot : toujours 0
accuracy_score(Y_test, Y_zero)*100

99.86959433264019

In [17]:
print("nbr de prdections predit",Y_pred.sum())
print("nbr de predection d'origine ",Y_test.sum())

nbr de prdections predit 46
nbr de predection d'origine  74


accuracy ne dit pas si ton modèle est bon, donc il faut le juger autrement. Trois étapes, dans cet ordre :

* La matrice de confusion : compter les 4 types de résultats (bonnes alertes, fausses alertes, fraudes ratées, normales bien reconnues).
* La précision et le rappel : transformer ces comptes en deux pourcentages qui répondent aux deux vraies questions.
* L’interprétation : dire en une phrase si le modèle est utile, et pourquoi.

In [18]:
# Etape1 la matrice d confusion
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(Y_test,Y_pred)
print(cm)

[[56667     5]
 [   33    41]]


In [19]:
# calcule de pécision et le rappel pour que on peut guguer le module
from sklearn.metrics import precision_score,recall_score
print("Precision:",precision_score(Y_test,Y_pred)*100)
print("rappel:",recall_score(Y_test,Y_pred)*100)


Precision: 89.13043478260869
rappel: 55.4054054054054


* Précision (89 %) : sur 100 alertes du modèle, environ 89 sont de vraies fraudes (je dirais 89, pas 90). Il ne crie à la fraude que lorsqu’il est très sûr.
* Rappel (55 %) : sur 100 fraudes qui existent vraiment, il en signale environ 55. Les 45 autres passent sans alerte <br>
=> La conséquence. C’est un bon modèle de base (baseline), à améliorer en cherchant à faire monter le rappel sans trop perdre en précision.

### Tâche 6 : rendre le modèle moins timide avec class_weight="balanced"

mon modéle de base est tros purdent : précision 89% rappel 55% ilf faut donc mnt montrer le rappel la 1ere chose a faire et de changer la facon dont le module apprend <br>
class_weight="balanced" corrige cela : il donne un poids plus fort à la classe rare, inversement proportionnel à sa fréquence. Chaque fraude compte alors comme des centaines de transactions normales, et le modèle est puni beaucoup plus s’il en rate une.

In [20]:
#on ajoute mm tt on change sauf dan la creation des modules
modele_bal=LogisticRegression(max_iter=1000,class_weight="balanced")
modele_bal.fit(X_train,Y_train)
Y_pred_bal=modele_bal.predict(X_test)

In [21]:
def evaluer(Y_vrai, Y_predit):
    print(confusion_matrix(Y_vrai, Y_predit))
    print("précision :", precision_score(Y_vrai, Y_predit)*100)
    print("rappel    :", recall_score(Y_vrai, Y_predit)*100)
evaluer(Y_test,Y_pred_bal)

[[55067  1605]
 [    7    67]]
précision : 4.007177033492823
rappel    : 90.54054054054053


INterppretation:
* Base : trop timide (précision haute, rappel bas).
* Balanced : trop alarmiste (rappel haut, précision basse).

#### Tâche 7 : le seuil de décision
=> pour un modèle de classification qui donne des probabilités, le seuil est le premier réglage à explorer quand précision et rappel ne sont pas équilibrés

In [22]:
proba = modele_bal.predict_proba(X_test)[:, 1]
#predict_proba(X_test) renvoie un tableau à deux colonnes : la probabilité de la classe 0 et celle de la classe 1. Les deux additionnées font 1.
# [:, 1] garde seulement la 2e colonne, c’est-à-dire la probabilité de fraude. Le : veut dire « toutes les lignes », le 1 veut dire « la colonne n° 1 ».
proba.tolist()

[0.05041968378420578,
 0.036990848830751585,
 0.03387244101199686,
 0.29098411687537684,
 0.06224699715922372,
 0.14255954425105777,
 0.2942775555303732,
 0.001399832125707147,
 0.06801749250984603,
 0.03788704485883529,
 0.031736694274525704,
 0.016284948349619535,
 0.08945355372048715,
 0.33380727642585306,
 0.026398111374041133,
 0.03966497593026951,
 0.060850185247321796,
 0.05859258538517255,
 0.032504078328480665,
 0.0008493977743783044,
 0.05124327507629897,
 0.07140295312612573,
 0.16300096466048547,
 0.30980281471907783,
 0.018508251583501063,
 0.05837240608794428,
 0.08827582855950993,
 0.00016301954503453522,
 0.0031959212582900656,
 0.3635317980504836,
 0.21866027045374833,
 0.2780134180340385,
 0.054105187422404985,
 0.021411501815550297,
 0.3338740332040898,
 0.11324987202906889,
 0.020949158887953397,
 0.11507755625835452,
 0.003458360769741813,
 0.04051794661693342,
 0.012360009203769949,
 0.031515628058459054,
 0.7676352117703192,
 0.09478930825098503,
 0.0262242996338

In [23]:
for seuil in [0.5, 0.8, 0.95, 0.99, 0.999]:
    Y_seuil = (proba >= seuil).astype(int)
    print(seuil,
          "alertes :", Y_seuil.sum(),
          "précision :", round(precision_score(Y_test, Y_seuil)*100, 1),
          "rappel :", round(recall_score(Y_test, Y_seuil)*100, 1))

0.5 alertes : 1672 précision : 4.0 rappel : 90.5
0.8 alertes : 411 précision : 15.3 rappel : 85.1
0.95 alertes : 158 précision : 38.6 rappel : 82.4
0.99 alertes : 108 précision : 54.6 rappel : 79.7
0.999 alertes : 80 précision : 70.0 rappel : 75.7


In [27]:
#choisir le seuil proprement
cut = int(len(X_train) * 0.8)

X_fit, X_val = X_train.iloc[:cut], X_train.iloc[cut:]
Y_fit, Y_val = Y_train.iloc[:cut], Y_train.iloc[cut:]

print(len(X_fit), len(X_val), Y_fit.sum(), Y_val.sum())

181584 45396 347 52


In [32]:
#Entraîne le modèle sur la partie apprentissage et calcule les probabilités sur la validatio
m = LogisticRegression(max_iter=1000, class_weight="balanced")
m.fit(X_fit, Y_fit)
proba_val = m.predict_proba(X_val)[:, 1]


In [36]:
print(proba_val[:5])
# on voir 5 nombres entre 0 et 1, car  Le modèle n’a pas encore dit « fraude » ou « normale ». C’est le seuil qui décide.
for s in [0.5, 0.8, 0.9, 0.95, 0.99, 0.999]:
    Y_pred_s = (proba_val >= s).astype(int)
    print("seuil :", s,
          "|| alertes :", Y_pred_s.sum(),
          "|| précision :", round(precision_score(Y_val, Y_pred_s, zero_division=0)*100, 1),
          "|| rappel :", round(recall_score(Y_val, Y_pred_s)*100, 1))

[0.01651185 0.13048791 0.07463308 0.00160209 0.01713433]
seuil : 0.5 || alertes : 1418 || précision : 3.2 || rappel : 88.5
seuil : 0.8 || alertes : 435 || précision : 10.3 || rappel : 86.5
seuil : 0.9 || alertes : 263 || précision : 16.3 || rappel : 82.7
seuil : 0.95 || alertes : 163 || précision : 25.8 || rappel : 80.8
seuil : 0.99 || alertes : 93 || précision : 43.0 || rappel : 76.9
seuil : 0.999 || alertes : 75 || précision : 53.3 || rappel : 76.9


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

prec, rap, seuils = precision_recall_curve(Y_val, proba_val)
# prec et rap ont 1 élément de plus que seuils (dernier point : précision=1, rappel=0)
f1 = 2*prec[:-1]*rap[:-1] / (prec[:-1] + rap[:-1] + 1e-12)
# c) meilleur seuil
i = f1.argmax()
seuil_choisi = seuils[i]
print(seuil_choisi, prec[i], rap[i], f1[i])

0.9999984273743361 0.9411764705882353 0.6153846153846154 0.7441860465111498


In [ ]:
# d) évaluation finale sur le test, une seule fois
m_final = LogisticRegression(max_iter=1000, class_weight="balanced")
m_final.fit(X_train, Y_train)                        # réentraîné sur tout le train
proba_test = m_final.predict_proba(X_test)[:, 1]
Y_final = (proba_test >= seuil_choisi).astype(int)   # seuil décidé sur la validation
evaluer(Y_test, Y_final)
print("PR-AUC :", average_precision_score(Y_test, proba_test))

[[56671     1]
 [   26    48]]
précision : 97.95918367346938
rappel    : 64.86486486486487
PR-AUC : 0.750962040376524


### Tâche 8 : tester si Hour améliore le modèle

In [44]:
import numpy as np
from sklearn.metrics import precision_recall_curve, average_precision_score

for d in (train, test):
    d["Hour"]     = (d["Time"] // 3600) % 24
    d["Hour_sin"] = np.sin(2*np.pi*d["Hour"]/24)
    d["Hour_cos"] = np.cos(2*np.pi*d["Hour"]/24)

In [45]:

def tester(cols):
    Xf, Xv = train.iloc[:cut][cols], train.iloc[cut:][cols]
    Yf, Yv = Y_train.iloc[:cut], Y_train.iloc[cut:]
    m = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xf, Yf)
    p = m.predict_proba(Xv)[:, 1]
    prec, rap, seuils = precision_recall_curve(Yv, p)
    f1 = 2*prec[:-1]*rap[:-1] / (prec[:-1] + rap[:-1] + 1e-12)
    return average_precision_score(Yv, p), f1.max()

sans = colonnes                           # V1..V28 + Amount_s
avec = colonnes + ["Hour_sin", "Hour_cos"]
print("sans Hour (PR-AUC, F1 max) :", tester(sans))
print("avec Hour (PR-AUC, F1 max) :", tester(avec))

sans Hour (PR-AUC, F1 max) : (0.7413727350225361, np.float64(0.7441860465111498))
avec Hour (PR-AUC, F1 max) : (0.7277729588004056, np.float64(0.7291666666661701))
